# 06. Qwen2.5:7b Inference
Dùng Qwen2.5:7b (qua Ollama) để fix các nhóm lỗi từ baseline lookup.

**Pipeline:**
1. Setup
2. Version 1: LLM thuần (không ViSoLex)
3. Version 2: LLM + ViSoLex dictionary
4. So sánh V1 vs V2
5. Xử lý over_change
6. Kết hợp & tính ERR trên toàn bộ validation set

## 1. Setup

In [ ]:
!pip install ollama
import ollama
import pandas as pd
import json
import ast

### 1.1 Load data

In [ ]:
# Load error analysis data
df = pd.read_csv("../outputs/error_wrong_tokens_with_meta.csv")

# Load validation set (dùng để lấy raw_tokens làm context)
val_df = pd.read_csv("../outputs/val_lookup_all_results.csv")
val_df["raw_tokens"] = val_df["raw_tokens"].apply(ast.literal_eval)

# Load validation set token level (dùng để tính ERR)
val_token_df = pd.read_csv("../outputs/val_lookup_token_level_results.csv")

print(f"df shape: {df.shape}")
print(f"val_df shape: {val_df.shape}")
print(f"val_token_df shape: {val_token_df.shape}")

### 1.2 Load ViSoLex dictionary

In [ ]:
# Download ViSoLex dictionary (chỉ cần chạy 1 lần)
# !curl -L https://raw.githubusercontent.com/HaDung2002/visolex/main/dict/dictionary.json -o ../data/visolex_dict.json

with open("../data/visolex_dict.json", "r", encoding="utf-8") as f:
    visolex_dict = json.load(f)

# Tách thành simple (1 nghĩa) và multi (nhiều nghĩa)
simple_lookup = {}  # token chỉ có 1 nghĩa → dùng luôn, không cần LLM
multi_lookup  = {}  # token có nhiều nghĩa → cần context để chọn đúng

for nsw, data in visolex_dict.items():
    normalized_list = data["normalized"]
    if len(normalized_list) == 1:
        simple_lookup[nsw] = normalized_list[0]
    else:
        multi_lookup[nsw] = normalized_list

print(f"ViSoLex entries: {len(visolex_dict)}")
print(f"Token 1 nghĩa (simple_lookup): {len(simple_lookup)}")
print(f"Token nhiều nghĩa (multi_lookup): {len(multi_lookup)}")

### 1.3 Filter các nhóm lỗi cần xử lý

In [ ]:
target_groups = ["oov", "rare_seen_token", "other_need_norm_error", "ambiguous_low_conf"]
target_df = df[df["error_group"].isin(target_groups)].copy()

print(f"Tổng token cần xử lý: {len(target_df)}")
print(target_df["error_group"].value_counts())

## 2. Version 1: LLM thuần (không ViSoLex)

### 2.1 Định nghĩa hàm normalize

In [ ]:
def normalize_llm_only(raw_token, sent_idx=None, val_df=None):
    raw_sentence = ""
    if val_df is not None and sent_idx is not None:
        raw_sentence = " ".join(val_df.loc[sent_idx, "raw_tokens"])

    if raw_sentence:
        content = f"""Câu: "{raw_sentence}"
Token cần chuẩn hóa: "{raw_token}"
Hãy trả về dạng chuẩn của token đó trong ngữ cảnh câu trên."""
    else:
        content = raw_token

    response = ollama.chat(
        model="qwen2.5:7b",
        messages=[
            {
                "role": "system",
                "content": """Bạn là hệ thống chuẩn hóa văn bản tiếng Việt mạng xã hội.
Nhiệm vụ: dựa vào ngữ cảnh câu, chuyển token viết tắt về dạng chuẩn tiếng Việt.
Chỉ trả về đúng 1 token đã chuẩn hóa, không giải thích, không thêm gì khác.

Ví dụ:
- ko → không
- dc → được
- t (trong câu "t đi học") → tao
- t (trong câu "t nghĩ vậy") → tớ"""
            },
            {"role": "user", "content": content}
        ]
    )
    return response["message"]["content"].strip(), "llm_only"

### 2.2 Chạy inference V1

In [ ]:
results_inference_v1 = []

for idx, row in target_df.iterrows():
    raw_token = row["raw_token"]
    pred_llm, source = normalize_llm_only(
        raw_token,
        sent_idx=row["sent_idx"],
        val_df=val_df
    )

    results_inference_v1.append({
        "sent_idx": row["sent_idx"],
        "token_pos": row["token_pos"],
        "raw_token": raw_token,
        "gold_token": row["gold_token"],
        "pred_lookup": row["pred_token"],
        "pred_llm": pred_llm,
        "source": source,
        "error_group": row["error_group"],
        "is_correct": pred_llm == row["gold_token"]
    })
    print(f"[{idx}] {raw_token} → {pred_llm} (gold: {row['gold_token']}) [{row['error_group']}]")

results_inference_v1_df = pd.DataFrame(results_inference_v1)
results_inference_v1_df.to_csv("../outputs/results_inference_v1.csv", index=False, encoding="utf-8-sig")

print(f"\nĐộ chính xác tổng: {results_inference_v1_df['is_correct'].mean():.2%}")
print(f"\nTheo error_group:")
print(results_inference_v1_df.groupby("error_group")["is_correct"].agg(["sum", "count", "mean"]).round(3))

## 3. Version 2: LLM + ViSoLex

### 3.1 Định nghĩa hàm normalize có candidates

In [ ]:
def normalize_with_candidates(raw_token, sent_idx=None, val_df=None):
    # Lấy câu gốc từ val_df
    raw_sentence = ""
    if val_df is not None and sent_idx is not None:
        raw_sentence = " ".join(val_df.loc[sent_idx, "raw_tokens"])

    # Nếu token có trong simple_lookup → dùng luôn, không cần LLM
    if raw_token in simple_lookup:
        return simple_lookup[raw_token], "simple_lookup"

    # Nếu token có nhiều nghĩa → dùng LLM chọn dựa trên context
    candidates = multi_lookup.get(raw_token, None)

    if candidates and raw_sentence:
        content = f"""Câu: "{raw_sentence}"
Token: "{raw_token}"
Các nghĩa có thể: {", ".join(candidates)}
Chỉ trả về đúng 1 từ phù hợp nhất với ngữ cảnh, không giải thích."""
    elif raw_sentence:
        content = f"""Câu: "{raw_sentence}"
Token cần chuẩn hóa: "{raw_token}"
Chỉ trả về đúng 1 token đã chuẩn hóa, không giải thích."""
    else:
        content = raw_token

    response = ollama.chat(
        model="qwen2.5:7b",
        messages=[
            {
                "role": "system",
                "content": """Bạn là hệ thống chuẩn hóa văn bản tiếng Việt mạng xã hội.
Nhiệm vụ: dựa vào ngữ cảnh câu, chọn dạng chuẩn phù hợp nhất cho token.
Nếu token đã đúng, giữ nguyên.
Chỉ trả về đúng 1 token, không giải thích."""
            },
            {"role": "user", "content": content}
        ]
    )
    return response["message"]["content"].strip(), "llm"

### 3.2 Chạy inference V2

In [ ]:
results_inference_v2 = []

for idx, row in target_df.iterrows():
    raw_token = row["raw_token"]
    pred_llm, source = normalize_with_candidates(
        raw_token,
        sent_idx=row["sent_idx"],
        val_df=val_df
    )

    results_inference_v2.append({
        "sent_idx": row["sent_idx"],
        "token_pos": row["token_pos"],
        "raw_token": raw_token,
        "gold_token": row["gold_token"],
        "pred_lookup": row["pred_token"],
        "pred_llm": pred_llm,
        "source": source,
        "error_group": row["error_group"],
        "is_correct": pred_llm == row["gold_token"]
    })
    print(f"[{idx}] {raw_token} → {pred_llm} (gold: {row['gold_token']}) [{row['error_group']}]")

results_inference_v2_df = pd.DataFrame(results_inference_v2)
results_inference_v2_df.to_csv("../outputs/results_inference_v2.csv", index=False, encoding="utf-8-sig")

print(f"\nĐộ chính xác tổng: {results_inference_v2_df['is_correct'].mean():.2%}")
print(f"\nTheo error_group:")
print(results_inference_v2_df.groupby("error_group")["is_correct"].agg(["sum", "count", "mean"]).round(3))

## 4. So sánh V1 vs V2

In [ ]:
print("=== SO SÁNH V1 vs V2 ===")
compare = pd.DataFrame({
    "v1_llm_only": results_inference_v1_df.groupby("error_group")["is_correct"].mean(),
    "v2_llm_visolex": results_inference_v2_df.groupby("error_group")["is_correct"].mean()
}).round(3)
compare["improvement"] = (compare["v2_llm_visolex"] - compare["v1_llm_only"]).round(3)
print(compare)

## 5. Xử lý nhóm over_change

**Nhận xét:** gold_token = raw_token (100%) → annotation nói không cần normalize.
Tuy nhiên pred_token (lookup baseline) đang normalize đúng về mặt ngữ nghĩa tiếng Việt.
→ Giữ nguyên pred_token thay vì raw_token.

In [ ]:
over_change_df = df[df["error_group"] == "over_change"].copy()

# Check raw_token == gold_token
match = (over_change_df["raw_token"] == over_change_df["gold_token"]).sum()
print(f"raw_token == gold_token: {match}/{len(over_change_df)} = {match/len(over_change_df):.2%}")

# Giữ nguyên pred_token (baseline lookup)
over_change_inference_df = over_change_df.copy()
over_change_inference_df["pred_final"] = over_change_inference_df["pred_token"]
over_change_inference_df["source"] = "keep_pred_lookup"
over_change_inference_df["is_correct"] = over_change_inference_df["pred_final"] == over_change_inference_df["gold_token"]

over_change_inference_df.to_csv("../outputs/over_change_inference.csv", index=False, encoding="utf-8-sig")

print(f"\nĐộ chính xác: {over_change_inference_df['is_correct'].mean():.2%}")
print("(0% là hợp lý vì gold_token = raw_token, nhưng pred_token đúng về ngữ nghĩa tiếng Việt)")

## 6. Kết hợp & tính ERR trên toàn bộ validation set

### 6.1 Gộp kết quả 5 nhóm lỗi

In [ ]:
# Thêm cột pred_final cho V2 (từ pred_llm)
results_inference_v2_df["pred_final"] = results_inference_v2_df["pred_llm"]

# Gộp tất cả 5 nhóm
all_results_inference_df = pd.concat([
    results_inference_v2_df[["sent_idx", "token_pos", "raw_token", "gold_token",
                              "pred_lookup", "pred_final", "source", "error_group", "is_correct"]],
    over_change_inference_df[["sent_idx", "token_pos", "raw_token", "gold_token",
                               "pred_token", "pred_final", "source", "error_group", "is_correct"]]
                               .rename(columns={"pred_token": "pred_lookup"})
], ignore_index=True)

all_results_inference_df.to_csv("../outputs/all_results_inference.csv", index=False, encoding="utf-8-sig")

print(f"Tổng token đã xử lý: {len(all_results_inference_df)}")
print(f"\nĐộ chính xác tổng: {all_results_inference_df['is_correct'].mean():.2%}")
print(f"\nTheo error_group:")
print(all_results_inference_df.groupby("error_group")["is_correct"].agg(["sum", "count", "mean"]).round(3))

### 6.2 Merge vào toàn bộ validation set

In [ ]:
# Merge pred_final từ inference vào val_token_df
val_full_df = val_token_df.merge(
    all_results_inference_df[["sent_idx", "token_pos", "pred_final"]],
    on=["sent_idx", "token_pos"],
    how="left"
)

# Token không có trong inference → dùng pred_token (lookup baseline)
val_full_df["pred_final"] = val_full_df["pred_final"].fillna(val_full_df["pred_token"])

print(f"Tổng token validation: {len(val_full_df)}")
print(val_full_df[["raw_token", "gold_token", "pred_token", "pred_final"]].head(5))

### 6.3 Tính ERR

Công thức: `ERR = (TP - FP) / (TP + FN) * 100`

- **TP**: cần normalize, system đúng
- **FP**: không cần normalize, system sửa
- **TN**: không cần normalize, system giữ nguyên
- **FN**: cần normalize, system sai hoặc bỏ qua

In [ ]:
TP = FP = TN = FN = 0

for _, row in val_full_df.iterrows():
    needs_norm = row["needs_normalization"]
    raw  = row["raw_token"]
    gold = row["gold_token"]
    pred = row["pred_final"]

    system_normalized = (pred != raw)
    correct = (pred == gold)

    if needs_norm == 0 and not system_normalized:
        TN += 1
    elif needs_norm == 0 and system_normalized:
        FP += 1
    elif needs_norm == 1 and correct:
        TP += 1
    elif needs_norm == 1 and not correct:
        FN += 1

ERR = (TP - FP) / (TP + FN) * 100

print(f"TP: {TP}")
print(f"FP: {FP}")
print(f"TN: {TN}")
print(f"FN: {FN}")
print(f"\nERR: {ERR:.2f}%")